## Workspace setup

In [1]:
from datetime import datetime  
import uproot
from functools import partial
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

import importlib

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys
sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")

# Change directory to the working directory
import os
os.chdir('/scratch_hdd/akalinow/ELITPC/PythonAnalysis/')

2025-12-11 09:46:45.817616: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Training dataset preparation

In [2]:
import io_functions as io
importlib.reload(io)

import plotting_functions as plf
importlib.reload(plf)

import utility_functions as utils
importlib.reload(utils)

batchSize = 64
dataset = tf.data.Dataset.load('MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm', compression="GZIP")
dataset = dataset.batch(batchSize, drop_remainder=True)
dataset = dataset.map(lambda x: x['sim'])
dataset = dataset.map(lambda x,y: (tf.reshape(x, (-1,)+io.projections.shape), tf.reshape(y, (-1,3,3))))
dataset = dataset.map(lambda x,y: (x, utils.XYZtoUVWT_event(y)))
dataset = dataset.map(lambda x,y: (x, tf.reshape(y, (-1,12))))
dataset = dataset.take(1000).cache('/scratch_ssd/akalinow/data_cache/MergedEvent_Track3D_TwoProng_gun_MC_200k_filtered_length_30-100mm_UVWT_cache').prefetch(tf.data.AUTOTUNE)
tfds.benchmark(dataset)

2025-12-11 09:46:56.494630: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-11 09:46:56.496998: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-12-11 09:46:56.569155: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:998] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-


************ Summary ************



  0%|          | 0/1000 [00:00<?, ?it/s]

Examples/sec (First included) 4.76 ex/sec (total: 1001 ex, 210.17 sec)
Examples/sec (First only) 2.48 ex/sec (total: 1 ex, 0.40 sec)
Examples/sec (First excluded) 4.77 ex/sec (total: 1000 ex, 209.77 sec)


2025-12-11 09:50:28.925313: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,duration,num_examples,avg
first+lasts,210.174717,1001,4.762704
first,0.403545,1,2.478041
lasts,209.771172,1000,4.767099


In [3]:
tfds.benchmark(dataset.take(100))


************ Summary ************



  0%|          | 0/100 [00:00<?, ?it/s]

Examples/sec (First included) 7.46 ex/sec (total: 101 ex, 13.55 sec)
Examples/sec (First only) 6.17 ex/sec (total: 1 ex, 0.16 sec)
Examples/sec (First excluded) 7.47 ex/sec (total: 100 ex, 13.38 sec)


2025-12-11 09:55:14.648117: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,duration,num_examples,avg
first+lasts,13.545089,101,7.456577
first,0.162185,1,6.165786
lasts,13.382904,100,7.472220


In [5]:
dataset.cardinality().numpy()

1000

## Model definition

In [4]:
def getModel():

  model = tf.keras.Sequential([
  tf.keras.layers.Input(shape=(256,512,3), name="input_image", dtype=tf.float32),
  tf.keras.layers.Resizing(height=256, width=256), 
  tf.keras.layers.Conv2D(16, 4, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(32, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.MaxPooling2D(),
  tf.keras.layers.Conv2D(64, 2, padding='same', activation='relu', 
                          data_format="channels_last"),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(16, activation='relu',
                          kernel_initializer=tf.keras.initializers.HeNormal(),
                          bias_initializer=tf.keras.initializers.RandomUniform(minval=-1, maxval=1)),
  tf.keras.layers.Dense(12, activation="linear")
  ])

  initial_learning_rate = 0.01
  lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=836,
                  decay_rate=0.98,
                  staircase=False)

  optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
  model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse', 'mape']) 

  model.summary()
  return model

## Model training

In [ ]:
%%time

import plotting_functions as plf
importlib.reload(plf)

log_dir = "logs/fit/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1, profile_batch=(10, 20))
early_stop_callback = tf.keras.callbacks.EarlyStopping(patience=2, verbose=1)
callbacks =  [early_stop_callback, tensorboard_callback]


model = getModel()
model.trainable = True

initial_learning_rate = 0.001
decay_steps = dataset.cardinality().numpy()
if decay_steps == tf.data.INFINITE_CARDINALITY or decay_steps == tf.data.UNKNOWN_CARDINALITY:
    decay_steps = 1000

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate,
                  decay_steps=decay_steps,
                  decay_rate=0.98,
                  staircase=False)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule) 
model.compile(optimizer = optimizer, 
                loss = 'mse', 
                metrics=['mse']) 


epochs=2
history = model.fit(dataset.skip(10),
                    epochs=epochs,
                    verbose = 1,
                    validation_data = dataset.take(10),
                    callbacks=callbacks
                    )
plf.plotTrainHistory(history)

current_time = datetime.now().strftime("%Y_%b_%d_%H_%M_%S")
print("Training start. Current Time =", current_time)

job_dir = f"training/{epochs:04d}_"+current_time+".keras"
model.save(job_dir)

job_dir = f"training/{epochs:04d}_"+current_time+"/"
model.export(job_dir)

2025-12-11 09:57:22.011300: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:104] Profiler session initializing.
2025-12-11 09:57:22.011318: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:119] Profiler session started.
2025-12-11 09:57:22.011369: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1239] Profiler found 2 GPUs
2025-12-11 09:57:22.092701: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:131] Profiler session tear down.
2025-12-11 09:57:22.092781: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1364] CUPTI activity buffer flushed


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resizing (Resizing)             │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 256, 256, 16)   │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 128, 128, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 262144)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │     4,194,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 12)             │           204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,206,732 (16.05 MB)

 Trainable params: 4,206,732 (16.05 MB)

...

 10/100 ━━━━━━━━━━━━━━━━━━━━ 9s 108ms/step - loss: 16541.7910 - mse: 16541.7910

2025-12-11 09:57:34.551723: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:104] Profiler session initializing.
2025-12-11 09:57:34.551744: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:119] Profiler session started.


 21/100 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - loss: 16495.2656 - mse: 16495.2656

2025-12-11 09:57:35.734533: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:70] Profiler session collecting data.
2025-12-11 09:57:35.738994: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1364] CUPTI activity buffer flushed
2025-12-11 09:57:35.746390: I external/local_xla/xla/backends/profiler/gpu/cupti_collector.cc:540]  GpuTracer has collected 1153 callback api events and 1116 activity events. 
2025-12-11 09:57:35.753358: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:131] Profiler session tear down.
2025-12-11 09:57:35.753995: I external/local_tsl/tsl/profiler/rpc/client/save_profile.cc:144] Collecting XSpace to repository: logs/fit/20251211-095722/plugins/profile/2025_12_11_09_57_35/hepzc62.xplane.pb


100/100 ━━━━━━━━━━━━━━━━━━━━ 24s 126ms/step - loss: 14928.7275 - mse: 14928.7275 - val_loss: 6615.3853 - val_mse: 6615.3853
Epoch 2/2
 73/100 ━━━━━━━━━━━━━━━━━━━━ 2s 105ms/step - loss: 5828.8306 - mse: 5828.8306

## Model performance on training data.

Fill Pandas DataFrame with true and response values.

In [ ]:
%%time
import utility_functions as utils
importlib.reload(utils)
import pandas as pd

nBatches = 10_000
data = np.zeros_like(utils.getSimRecoColumns(utils.columnsUVWT).reshape(1,-1)) 

for aBatch in dataset.take(nBatches):

    features = aBatch[0]
    labels = aBatch[1].numpy()
    modelAnswer = model(features).numpy()
    data = np.append(data, np.column_stack((labels,modelAnswer)), axis=0)

df = pd.DataFrame(data=data[1:], columns = utils.getSimRecoColumns(utils.columnsUVWT), dtype=np.float32)
df.describe()    

### Resolution plots

In [ ]:
import plotting_functions as plf
importlib.reload(plf)

#plf.controlPlots(df)
plf.plotEndPointRes(df=df, edge="Vtx", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Alpha", coordinates=["u", "v", "w", "t"])
plf.plotEndPointRes(df=df, edge="Carbon", coordinates=["u", "v", "w", "t"])